In [1]:
import mlflow
import pandas as pd
from mlflow import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

In [2]:
from src.common.config import get_settings
from src.common.features import TARGET_COLUMN, load_dataframe, prepare_dataframe, to_model_records
from src.common.model_registry import configure_tracking, ensure_experiment

In [4]:
settings = get_settings()
prepared = prepare_dataframe(load_dataframe(settings.train_data_uri), include_target= True)
X_train, X_valid, y_train, y_valid = train_test_split(
    to_model_records(prepared), prepared[TARGET_COLUMN], test_size= 0.2, random_state= 42
)

len(X_train), len(X_valid)

(35620, 8905)

RMSE: 4.80


In [9]:
configure_tracking(settings.mlflow_tracking_uri)
experiment_id = ensure_experiment(settings.experiment_name)
mlflow.set_experiment(settings.experiment_name)

candidates = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=50,max_depth=20, n_jobs=-1, random_state=42),
}

results = []

for name,model in candidates.items():
    print(name,model)

    with mlflow.start_run(experiment_id=experiment_id, run_name=f"ext-{name}") as run:
        pipeline = make_pipeline(DictVectorizer(), model)
        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_valid)
        rmse = root_mean_squared_error(y_valid, predictions)

        mlflow.log_param("feature_columns", "trip_route,trip_distance")
        mlflow.log_param("model_type", name)
        mlflow.log_metric("rmse", float(rmse))
        info = mlflow.sklearn.log_model(
            sk_model=pipeline,
            name=settings.model_artifact_path,
            registered_model_name=settings.registered_model_name,
            serialization_format="skops",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
    print(f"{name}: RMSE {rmse:.2f} -> version {info.registered_model_version}")
    results.append({"model": name, "rmse": rmse,"version": info.registered_model_version})

comparison = pd.DataFrame(results).sort_values("rmse")
comparison
      

LinearRegression LinearRegression()


Registered model 'green-taxi-duration' already exists. Creating a new version of this model...
2026/09/30 20:23:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: green-taxi-duration, version 6
Created version '6' of model 'green-taxi-duration'.


🏃 View run ext-LinearRegression at: http://127.0.0.1:5001/#/experiments/1/runs/97818ff997714e369e93452ac0ba0a64
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
LinearRegression: RMSE 4.80 -> version 6
Ridge Ridge()


Registered model 'green-taxi-duration' already exists. Creating a new version of this model...
2026/09/30 20:23:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: green-taxi-duration, version 7
Created version '7' of model 'green-taxi-duration'.


🏃 View run ext-Ridge at: http://127.0.0.1:5001/#/experiments/1/runs/a574285d60c341a9993541916c000e64
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
Ridge: RMSE 4.67 -> version 7
RandomForestRegressor RandomForestRegressor(max_depth=20, n_estimators=50, n_jobs=-1, random_state=42)


Registered model 'green-taxi-duration' already exists. Creating a new version of this model...
2026/09/30 20:24:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: green-taxi-duration, version 8
Created version '8' of model 'green-taxi-duration'.


🏃 View run ext-RandomForestRegressor at: http://127.0.0.1:5001/#/experiments/1/runs/b5e7297ae8204854ae4151618ecdf969
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
RandomForestRegressor: RMSE 4.67 -> version 8


,model,rmse,version
2,RandomForestRegressor,4.667118,8
1,Ridge,4.673217,7
0,LinearRegression,4.800469,6


In [10]:
client = MlflowClient()
client.set_registered_model_alias(settings.registered_model_name, "champion", "7")